# Simple GenAI App using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
#langsmith tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true" #requirement of langchain, to enable tracing
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

Load Data --> Docs. Divide docs into chunks. Convert chunks into Vectors using vector embeddings. Store Vectors in Vectorstore DB.

In [2]:
#Data Ingestion -> from the website we need to scrape the data 

from langchain_community.document_loaders import WebBaseLoader


/Users/user/Desktop/genai/venv/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-concepts?_gl=1*xsd98z*_gcl_au*MTY4NjcyNjE0NS4xNzY0NTk1OTE0*_ga*MTk3OTc1MjgzLjE3NjQ1NjYzNTQ.*_ga_47WX3HKKY2*czE3NjQ1OTU5MTQkbzEkZzAkdDE3NjQ1OTU5MTQkajYwJGwwJGgw")
loader 

In [4]:
docs = loader.load()#scrapes all content from given link and assigns to 'docs' variable
print(docs)

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts?_gl=1*xsd98z*_gcl_au*MTY4NjcyNjE0NS4xNzY0NTk1OTE0*_ga*MTk3OTc1MjgzLjE3NjQ1NjYzNTQ.*_ga_47WX3HKKY2*czE3NjQ1OTU5MTQkbzEkZzAkdDE3NjQ1OTU5MTQkajYwJGwwJGgw', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='Observability concepts - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationObservability conceptsGet startedObservabilityEvaluationPrompt engineeringDeploymentAgent BuilderPlatform setupOverviewQuickstartConceptsTrace a RAG applicationTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesQuery traces (SDK)Compare tracesShare or unshare a trace publiclyView server logs for a traceBulk export trace da

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents = splitter.split_documents(docs)

In [6]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts?_gl=1*xsd98z*_gcl_au*MTY4NjcyNjE0NS4xNzY0NTk1OTE0*_ga*MTk3OTc1MjgzLjE3NjQ1NjYzNTQ.*_ga_47WX3HKKY2*czE3NjQ1OTU5MTQkbzEkZzAkdDE3NjQ1OTU5MTQkajYwJGwwJGgw', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='Observability concepts - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationObservability conceptsGet startedObservabilityEvaluationPrompt engineeringDeploymentAgent BuilderPlatform setupOverviewQuickstartConceptsTrace a RAG applicationTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesQuery traces (SDK)Compare tracesShare or unshare a trace publiclyView server logs for a traceBulk export trace da

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [8]:
from langchain_community.vectorstores import FAISS

In [9]:
vector_db = FAISS.from_documents(documents,embeddings)


In [10]:
#query from a vector db

query = "feedback entry consists of a feedback tag and feedback score"
result = vector_db.similarity_search(query)
result[0].page_content

'\u200bFeedback\nFeedback allows you to score an individual run based on certain criteria. Each feedback entry consists of a feedback tag and feedback score, and is bound to a run by a unique run ID. Feedback can be continuous or discrete (categorical), and you can reuse feedback tags across different runs within an organization.\nYou can collect feedback on runs in a number of ways:\n\nSent up along with a trace from the LLM application.\nGenerated by a user in the app inline or in an annotation queue.\nGenerated by an automatic evaluator during offline evaluation.\nGenerated by an online evaluator.\n\nTo learn more about how feedback is stored in the application, refer to the Feedback data format guide.\n\n\u200bTags\nTags are collections of strings that can be attached to runs. You can use tags to do the following in the LangSmith UI:\n\nCategorize runs for easier search.\nFilter runs.\nGroup runs together for analysis.\n\nLearn how to attach tags to your traces.'

In [11]:
import langchain
print(langchain.__version__)


1.0.5


In [12]:

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [13]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")


We use a document chain to combine retrieved document chunks into a single prompt so the LLM answers strictly from those chunks and not from outside knowledge. It ensures context-limited, grounded answers.

In [22]:
#context : #instead of searching entire page, just search in premises of given context like paragraph etc

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """
Answer the following questions based only on the provided context
<context>
{context} 
</context>

Question: {input}

"""
)

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    document_variable_name="context"   
)

In [15]:
from langchain_core.documents import Document

setting context manually:

In [23]:
document_chain.invoke(
    {
        "input":"Feedback can be continuous or discrete",
        "context":[Document(page_content="Feedback can be continuous or discrete (categorical), and you can reuse feedback tags across different runs within an organization")]
    }
)

'True, feedback can be continuous or discrete (categorical) based on the provided context.'

We want documents to first come from retriever we just set up. This way retreiver dynamically selelcts the most relevant documents and pass those in for a given question.

we convert vectoredb into retriever. Retreiver is an interface , where we pass input thru retriever and get response from vector store 

In [26]:
retriever = vector_db.as_retriever()


✅ What happens internally

User query → passed to retriever

The retriever searches your vector DB (FAISS / Chroma / PGVector etc.)

It returns relevant Document chunks

These documents are automatically passed into your document chain under the variable name "context" (or "documents" depending on the chain type)

The document chain stuffs the docs into the prompt and sends it to the LLM

So yes — after combining into a retrieval chain, you no longer provide context manually.
The retrieval chain fills the context/documents variable automatically.

In [27]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever,document_chain)


In [29]:
response = retrieval_chain.invoke({"input":"Feedback allows you to score an individual run?"})
response['answer']

'Yes, feedback allows you to score an individual run based on certain criteria.'

In [30]:
response

{'input': 'Feedback allows you to score an individual run?',
 'context': [Document(id='15dbcc5a-0242-4576-b155-cd14d3a13cae', metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts?_gl=1*xsd98z*_gcl_au*MTY4NjcyNjE0NS4xNzY0NTk1OTE0*_ga*MTk3OTc1MjgzLjE3NjQ1NjYzNTQ.*_ga_47WX3HKKY2*czE3NjQ1OTU5MTQkbzEkZzAkdDE3NjQ1OTU5MTQkajYwJGwwJGgw', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='\u200bFeedback\nFeedback allows you to score an individual run based on certain criteria. Each feedback entry consists of a feedback tag and feedback score, and is bound to a run by a unique run ID. Feedback can be continuous or discrete (categorical), and you can reuse feedback tags across different runs within an organization.\nYou can collect feedback on runs in a number of ways:\n\nSent up along with a trace from the LLM application.\nGenerated by a user in the app inline or in an annotation queue.\nGenerated by an automatic evaluator during 

In [31]:
response['context']

[Document(id='15dbcc5a-0242-4576-b155-cd14d3a13cae', metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts?_gl=1*xsd98z*_gcl_au*MTY4NjcyNjE0NS4xNzY0NTk1OTE0*_ga*MTk3OTc1MjgzLjE3NjQ1NjYzNTQ.*_ga_47WX3HKKY2*czE3NjQ1OTU5MTQkbzEkZzAkdDE3NjQ1OTU5MTQkajYwJGwwJGgw', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='\u200bFeedback\nFeedback allows you to score an individual run based on certain criteria. Each feedback entry consists of a feedback tag and feedback score, and is bound to a run by a unique run ID. Feedback can be continuous or discrete (categorical), and you can reuse feedback tags across different runs within an organization.\nYou can collect feedback on runs in a number of ways:\n\nSent up along with a trace from the LLM application.\nGenerated by a user in the app inline or in an annotation queue.\nGenerated by an automatic evaluator during offline evaluation.\nGenerated by an online evaluator.\n\nTo learn more a